<a href="https://colab.research.google.com/github/korkutanapa/OEE_ARTICLE_STUDIES/blob/main/TS_TDA_PREDICTION_TDA_STEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q "numpy==1.26.4" "scipy==1.11.4" "scikit-learn==1.3.2"
%pip install -q "giotto-tda==0.6.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
feature-engine 1.9.3 requires scikit-learn>=1.4.0, but you have scikit-learn 1.3.2 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.11.4 which is incompatible.
cuml-cu12 25.10.0 requires scikit-learn>=1.4, but you have scikit-learn 1.3.2 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.3.2 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
imbalanced-learn 0.14.0 requires scikit-learn<2,>=1.4.2, but you have scikit-learn 1.3.2 which is incompatible.


In [24]:
import pandas as pd
# Load the file for time series
data_path = 'tda_ready.xlsx'
df = pd.read_excel(data_path)

In [25]:
window_length = 24
window_size = 24
time_delay = 8
dimension = 3
stride = 1

In [26]:
import numpy as np
from gtda.time_series import SlidingWindow, TakensEmbedding
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import Scaler

# --- 1) Extract the time series as (n_timestamps, 1) ---
# df: your DataFrame, with a column 'Operation'
X = df['Operation'].to_numpy(dtype=float).reshape(-1, 1)   # shape: (n_timestamps, 1)

# --- 2) Sliding window: cut into overlapping windows ---
SW = SlidingWindow(size=window_size, stride=stride)
# Typical output shape: (n_windows, window_size, n_features)
X_sw = SW.fit_transform(X)

# For convenience, treat each window as an independent univariate series
n_windows, win_len, n_feat = X_sw.shape  # n_feat should be 1
X_windows = X_sw.reshape(n_windows, win_len)   # shape: (n_windows, window_size)

# --- 3) Takens embedding on each window ---
TE = TakensEmbedding(time_delay=time_delay, dimension=dimension)
# Input: (n_windows, window_size)
# Output: (n_windows, n_points, dimension)
X_te = TE.fit_transform(X_windows)

# --- 4) Vietoris–Rips persistence on embedded windows ---
VR = VietorisRipsPersistence(homology_dimensions=[0, 1])
# Input: (n_windows, n_points, dimension)
# Output: array of diagrams, one per window
diagrams = VR.fit_transform(X_te)

# --- 5) Scale persistence diagrams (normalization in diagram space) ---
scaler = Scaler()
scaled_diagrams = scaler.fit_transform(diagrams)


In [27]:
import numpy as np
import pandas as pd
from gtda.diagrams import (
    PersistenceEntropy,
    Amplitude,
    BettiCurve,
    PersistenceLandscape,
    Silhouette,
    HeatKernel,
)

# -------------------------------------------------------------------
# Helper: Carlsson features
# -------------------------------------------------------------------
def compute_carlsson_features(births, deaths, fn_max=5):
    """Compute Carlsson-type features f1..f5 for a single diagram."""
    if len(births) == 0:
        return [0.0] * fn_max

    lifetimes = deaths - births
    d_max = np.max(deaths) if len(deaths) > 0 else 0.0

    f1 = np.sum(births * lifetimes) if fn_max >= 1 else None
    f2 = np.sum((d_max - deaths) * lifetimes) if fn_max >= 2 else None
    f3 = np.sum((births ** 2) * (lifetimes ** 4)) if fn_max >= 3 else None
    f4 = np.sum(((d_max - deaths) ** 2) * (lifetimes ** 4)) if fn_max >= 4 else None
    f5 = np.max(lifetimes) if (fn_max >= 5 and len(lifetimes) > 0) else None

    values = [f1, f2, f3, f4, f5][:fn_max]
    return [v if v is not None else 0.0 for v in values]


# -------------------------------------------------------------------
# Helper: heat kernel feature extraction
# -------------------------------------------------------------------
def extract_heat_kernel_features(heat_kernel_features):
    """
    heat_kernel_features: array of shape
        (n_windows, 2, n_bins, n_bins) for H0 and H1

    Returns:
        DataFrame (n_windows, many_features)
    """
    feature_list = []

    for i in range(heat_kernel_features.shape[0]):  # per window
        sample_feats = {}

        for dim_idx, hom_dim in enumerate(["H0", "H1"]):
            matrix = heat_kernel_features[i, dim_idx, :, :]  # shape: (n_bins, n_bins)

            # Global L1/L2
            sample_feats[f"HK_L1_{hom_dim}"] = np.sum(np.abs(matrix))
            sample_feats[f"HK_L2_{hom_dim}"] = np.sqrt(np.sum(matrix ** 2))

            # Per-column L1/L2
            n_cols = matrix.shape[1]
            for col in range(n_cols):
                col_data = matrix[:, col]
                sample_feats[f"HK_Col{col}_L1_{hom_dim}"] = np.sum(np.abs(col_data))
                sample_feats[f"HK_Col{col}_L2_{hom_dim}"] = np.sqrt(
                    np.sum(col_data ** 2)
                )

        feature_list.append(sample_feats)

    heat_kernel_df = pd.DataFrame(feature_list)
    return heat_kernel_df


# -------------------------------------------------------------------
# Helper: Betti / landscape / silhouette norm features
# -------------------------------------------------------------------
def compute_curve_norm_features(curve_array, prefix):
    """
    curve_array: shape (n_windows, 2, n_points)   (H0, H1)
    prefix: string prefix for column names (e.g. 'betti', 'landscape', 'silhouette')

    Returns DataFrame with columns:
        {prefix}_L1_H0, {prefix}_L2_H0, {prefix}_L1_H1, {prefix}_L2_H1
    """
    n_windows = curve_array.shape[0]

    l1_h0, l2_h0, l1_h1, l2_h1 = [], [], [], []

    for i in range(n_windows):
        h0 = curve_array[i, 0, :]
        h1 = curve_array[i, 1, :]

        l1_h0.append(np.sum(np.abs(h0)))
        l2_h0.append(np.sqrt(np.sum(h0 ** 2)))

        l1_h1.append(np.sum(np.abs(h1)))
        l2_h1.append(np.sqrt(np.sum(h1 ** 2)))

    df = pd.DataFrame(
        {
            f"{prefix}_L1_H0": l1_h0,
            f"{prefix}_L2_H0": l2_h0,
            f"{prefix}_L1_H1": l1_h1,
            f"{prefix}_L2_H1": l2_h1,
        }
    )
    return df


# -------------------------------------------------------------------
# Helper: flatten simple vector features
# -------------------------------------------------------------------
def flatten_features_to_df(features_array, prefix):
    """
    features_array: np.ndarray of shape (n_windows, ...)
    prefix: base name for columns
    """
    flattened = features_array.reshape(features_array.shape[0], -1)
    cols = [f"{prefix}_{i}" for i in range(flattened.shape[1])]
    return pd.DataFrame(flattened, columns=cols)


# -------------------------------------------------------------------
# Helper: diagram statistics + Carlsson features
# -------------------------------------------------------------------
def extract_diagram_statistics(scaled_diagrams, fn_max=5):
    """
    scaled_diagrams: array-like of diagrams, each of shape (n_points, 3)
                     columns: [birth, death, homology_dim]
    Returns:
        features_df: DataFrame with per-window summary stats + Carlsson features.
    """
    n_windows = len(scaled_diagrams)
    features = {}

    for dim in [0, 1]:  # H0, H1
        # filter diagrams by dimension
        filtered_diagrams = [
            diagram[diagram[:, 2] == dim] for diagram in scaled_diagrams
        ]

        for i, diagram_dim in enumerate(filtered_diagrams):
            lifetimes = diagram_dim[:, 1] - diagram_dim[:, 0] if len(diagram_dim) > 0 else np.array([])
            sample_key = f"sample_{i}"

            if sample_key not in features:
                features[sample_key] = {}

            # Basic stats
            features[sample_key][f"dim_{dim}_num_features"] = len(lifetimes)
            features[sample_key][f"dim_{dim}_sum_lifetimes"] = float(np.sum(lifetimes))

            if len(lifetimes) > 0:
                features[sample_key][f"dim_{dim}_max_lifetime"] = float(np.max(lifetimes))
                features[sample_key][f"dim_{dim}_mean_lifetime"] = float(np.mean(lifetimes))
                features[sample_key][f"dim_{dim}_median_lifetime"] = float(np.median(lifetimes))
                features[sample_key][f"dim_{dim}_std_lifetime"] = float(np.std(lifetimes))
                features[sample_key][f"dim_{dim}_variance_lifetime"] = float(np.var(lifetimes))
                features[sample_key][f"dim_{dim}_min_lifetime"] = float(np.min(lifetimes))

                births = diagram_dim[:, 0]
                deaths = diagram_dim[:, 1]

                features[sample_key][f"dim_{dim}_sum_birth_times"] = float(np.sum(births))
                features[sample_key][f"dim_{dim}_mean_birth_time"] = float(np.mean(births))
                features[sample_key][f"dim_{dim}_sum_death_times"] = float(np.sum(deaths))
                features[sample_key][f"dim_{dim}_mean_death_time"] = float(np.mean(deaths))

                # Carlsson features
                f_vals = compute_carlsson_features(births, deaths, fn_max=fn_max)
                for j, fv in enumerate(f_vals, start=1):
                    features[sample_key][f"dim_{dim}_carlsson_f{j}"] = float(fv)
            else:
                # Empty diagram → set everything to 0
                for metric in [
                    "max_lifetime",
                    "mean_lifetime",
                    "median_lifetime",
                    "std_lifetime",
                    "variance_lifetime",
                    "min_lifetime",
                    "sum_birth_times",
                    "mean_birth_time",
                    "sum_death_times",
                    "mean_death_time",
                ]:
                    features[sample_key][f"dim_{dim}_{metric}"] = 0.0

                for j in range(1, fn_max + 1):
                    features[sample_key][f"dim_{dim}_carlsson_f{j}"] = 0.0

    features_df = pd.DataFrame.from_dict(features, orient="index")
    features_df.reset_index(drop=True, inplace=True)
    return features_df


# -------------------------------------------------------------------
# Main: build full windowed TDA feature matrix
# -------------------------------------------------------------------
def build_windowed_tda_feature_matrix(scaled_diagrams, heat_sigma=0.1, heat_bins=100, fn_max=5):
    """
    scaled_diagrams: array-like of persistence diagrams per window
    Returns:
        final_combined_df: DataFrame of all TDA features per window
    """

    # 1) Initialize feature extractors
    persistence_entropy = PersistenceEntropy()
    amplitude_bottleneck = Amplitude(metric="bottleneck")
    amplitude_wasserstein = Amplitude(metric="wasserstein")
    amplitude_landscape = Amplitude(metric="landscape")
    betti_curve = BettiCurve()
    persistence_landscape = PersistenceLandscape()
    silhouette = Silhouette()
    heat_kernel = HeatKernel(sigma=heat_sigma, n_bins=heat_bins, n_jobs=-1)

    # 2) Transform diagrams
    entropy_features = persistence_entropy.fit_transform(scaled_diagrams)
    amplitude_bottleneck_features = amplitude_bottleneck.fit_transform(scaled_diagrams)
    amplitude_wasserstein_features = amplitude_wasserstein.fit_transform(scaled_diagrams)
    amplitude_landscape_features = amplitude_landscape.fit_transform(scaled_diagrams)
    betti_features = betti_curve.fit_transform(scaled_diagrams)
    landscape_features = persistence_landscape.fit_transform(scaled_diagrams)
    silhouette_features = silhouette.fit_transform(scaled_diagrams)
    heat_kernel_features = heat_kernel.fit_transform(scaled_diagrams)

    # 3) Build DataFrames from each group

    # 3.1 Betti curves, landscapes, silhouettes → norm-based summaries
    betti_df = compute_curve_norm_features(betti_features, prefix="betti")
    landscape_df = compute_curve_norm_features(landscape_features, prefix="landscape")
    silhouette_df = compute_curve_norm_features(silhouette_features, prefix="silhouette")

    # 3.2 Flattenable features: entropy and amplitudes
    entropy_df = flatten_features_to_df(entropy_features, prefix="entropy")
    amplitude_bottleneck_df = flatten_features_to_df(
        amplitude_bottleneck_features, prefix="amplitude_bottleneck"
    )
    amplitude_wasserstein_df = flatten_features_to_df(
        amplitude_wasserstein_features, prefix="amplitude_wasserstein"
    )
    amplitude_landscape_df = flatten_features_to_df(
        amplitude_landscape_features, prefix="amplitude_landscape"
    )

    # 3.3 Heat kernel features
    heat_kernel_df = extract_heat_kernel_features(heat_kernel_features)

    # 4) Combine all "simple" features
    combined_df = pd.concat(
        [
            entropy_df,
            amplitude_bottleneck_df,
            amplitude_wasserstein_df,
            amplitude_landscape_df,
            betti_df,
            landscape_df,
            silhouette_df,
            heat_kernel_df,
        ],
        axis=1,
    )

    # 5) Add diagram statistics (lifetimes, birth/death stats, Carlsson)
    stats_df = extract_diagram_statistics(scaled_diagrams, fn_max=fn_max)
    final_combined_df = pd.concat(
        [combined_df.reset_index(drop=True), stats_df.reset_index(drop=True)], axis=1
    )

    # 6) Remove columns with NaN or ±inf anywhere
    mask_good = ~final_combined_df.isin([np.inf, -np.inf]).any() & final_combined_df.notna().all()
    final_combined_df = final_combined_df.loc[:, mask_good]

    return final_combined_df


In [28]:
final_combined_df = build_windowed_tda_feature_matrix(scaled_diagrams)

In [29]:
print(list(final_combined_df.columns))


['entropy_0', 'entropy_1', 'amplitude_bottleneck_0', 'amplitude_bottleneck_1', 'amplitude_wasserstein_0', 'amplitude_wasserstein_1', 'amplitude_landscape_0', 'amplitude_landscape_1', 'betti_L1_H0', 'betti_L2_H0', 'betti_L1_H1', 'betti_L2_H1', 'landscape_L1_H0', 'landscape_L2_H0', 'landscape_L1_H1', 'landscape_L2_H1', 'silhouette_L1_H0', 'silhouette_L2_H0', 'silhouette_L1_H1', 'silhouette_L2_H1', 'HK_L1_H0', 'HK_L2_H0', 'HK_Col0_L1_H0', 'HK_Col0_L2_H0', 'HK_Col1_L1_H0', 'HK_Col1_L2_H0', 'HK_Col2_L1_H0', 'HK_Col2_L2_H0', 'HK_Col3_L1_H0', 'HK_Col3_L2_H0', 'HK_Col4_L1_H0', 'HK_Col4_L2_H0', 'HK_Col5_L1_H0', 'HK_Col5_L2_H0', 'HK_Col6_L1_H0', 'HK_Col6_L2_H0', 'HK_Col7_L1_H0', 'HK_Col7_L2_H0', 'HK_Col8_L1_H0', 'HK_Col8_L2_H0', 'HK_Col9_L1_H0', 'HK_Col9_L2_H0', 'HK_Col10_L1_H0', 'HK_Col10_L2_H0', 'HK_Col11_L1_H0', 'HK_Col11_L2_H0', 'HK_Col12_L1_H0', 'HK_Col12_L2_H0', 'HK_Col13_L1_H0', 'HK_Col13_L2_H0', 'HK_Col14_L1_H0', 'HK_Col14_L2_H0', 'HK_Col15_L1_H0', 'HK_Col15_L2_H0', 'HK_Col16_L1_H0', 'HK

In [30]:
file_path = 'tda_features_from_residuals_all.xlsx'
final_combined_df.to_excel(file_path, index=False)

FEATURE SELECTION

In [31]:
# Dropping columns with low variance from final_combined_df
if 'final_combined_df' in locals():
    # Calculate variance of each column
    variance = final_combined_df.var()

    # Threshold for low variance (can be adjusted)
    low_variance_threshold = 0.01  # Example: close to zero variance

    # Identify columns with variance below the threshold
    low_variance_columns = variance[variance < low_variance_threshold].index

    # Drop low variance columns
    final_combined_df_cleaned = final_combined_df.drop(columns=low_variance_columns)

In [32]:
data=final_combined_df_cleaned

In [33]:
# Identify columns with inf, -inf, or NaN values
cols_with_invalid_values = data.columns[data.isin([np.inf, -np.inf]).any() | data.isna().any()]

# Replace inf and -inf with NaN first
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill NaN values with column mean
data[cols_with_invalid_values] = data[cols_with_invalid_values].apply(lambda col: col.fillna(col.mean()), axis=0)

In [34]:
simplified_df=data

In [35]:
# Adjust the indices of the feature dataset for regression analysis
regression_features_df = simplified_df[:-1].reset_index(drop=True)
regression_target_series = df['Operation'][window_size:].reset_index(drop=True)

# Combine features and target into a single dataset for regression analysis
regression_dataset = regression_features_df.copy()
regression_dataset['Target'] = regression_target_series

In [36]:
# Adjust the indices of the feature dataset for regression analysis
regression_features_df = simplified_df[:-1].reset_index(drop=True)
regression_target_series = df['Operation'][window_size:].reset_index(drop=True)

# Combine features and target into a single dataset for regression analysis
regression_dataset = regression_features_df.copy()
regression_dataset['Target'] = regression_target_series

In [37]:
df2=regression_dataset

In [28]:
# Install feature_engine (run this once in a cell)
%pip install feature_engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 94.4 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.2
    Uninstalling scikit-learn-1.3.2:
      Successfully uninstalled scikit-learn-1.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
giotto-tda 0.6.2 requires scikit-learn==1.3.2, but you have scikit-learn 1.7.2 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.11.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


In [38]:
from sklearn.ensemble import RandomForestRegressor
from feature_engine.selection import SmartCorrelatedSelection

In [39]:
# Assume the last column is the target variable (modify if needed)
target_column = "Target"  # Change if your target column has a different name
X = df2.drop(columns=[target_column])  # Extract feature columns
y = df2[target_column]  # Define the target variable

# Initialize SmartCorrelatedSelection for regression
tr = SmartCorrelatedSelection(
    variables=None,  # Apply to all numeric features
    method="pearson",  # Pearson correlation method
    threshold=0.99,  # Correlation threshold
    missing_values="raise",  # Raise error if missing values exist
    selection_method="model_performance",  # Use model performance to select features
    estimator=RandomForestRegressor(random_state=1, n_estimators=100),  # RandomForest for regression
    scoring="r2",  # Use R² as the evaluation metric
    cv=3  # 3-fold cross-validation
)

# Fit and transform the dataset to select best features
X_selected = tr.fit_transform(X, y)

# Reconstruct df2 with selected features and target variable
df2 = pd.concat([X_selected, y], axis=1)

In [40]:
from sklearn.preprocessing import MinMaxScaler

# Separate features and target
features = df2.drop(columns=["Target"])
target = df2["Target"]

# Apply scaling only to features
scaler = MinMaxScaler()
features_scaled = pd.DataFrame(scaler.fit_transform(features), columns=features.columns, index=features.index)

# Recombine scaled features with the original Target
df2_scaled = pd.concat([features_scaled, target], axis=1)

In [41]:
file_path = 'tda_features_from_residuals_selected_scaled.xlsx'
df2_scaled.to_excel(file_path, index=False)  # index=False to exclude the index column

In [42]:
df2_scaled

,entropy_0,entropy_1,amplitude_bottleneck_0,amplitude_wasserstein_0,betti_L1_H0,betti_L2_H0,betti_L1_H1,betti_L2_H1,landscape_L1_H0,silhouette_L1_H0,...,HK_Col95_L2_H1,HK_Col96_L1_H1,HK_Col97_L1_H1,HK_Col99_L2_H1,dim_0_median_lifetime,dim_0_std_lifetime,dim_0_carlsson_f2,dim_0_carlsson_f4,dim_1_sum_death_times,Target
0,0.939849,0.0,0.098614,0.159809,0.167857,0.289040,0.0,0.0,0.023583,0.030416,...,0.0,0.0,0.0,0.0,0.224633,0.047735,0.024870,0.000096,0.0,-36.630516
1,0.452725,0.0,0.675501,0.536220,0.389286,0.389639,0.0,0.0,0.490589,0.469574,...,0.0,0.0,0.0,0.0,0.248784,0.661514,0.427005,0.034430,0.0,-32.755274
2,0.502780,0.0,0.613014,0.488256,0.360714,0.379624,0.0,0.0,0.412899,0.384258,...,0.0,0.0,0.0,0.0,0.248784,0.589904,0.377790,0.025140,0.0,-16.242944
3,0.861676,0.0,0.270372,0.346603,0.339286,0.458352,0.0,0.0,0.103865,0.113383,...,0.0,0.0,0.0,0.0,0.269650,0.223178,0.124355,0.003521,0.0,6.891582
4,0.788548,0.0,0.420537,0.481523,0.453571,0.529010,0.0,0.0,0.214931,0.208197,...,0.0,0.0,0.0,0.0,0.284929,0.371510,0.258185,0.028176,0.0,-2.448256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
654,0.945979,0.0,0.475822,0.733742,0.767857,0.888381,0.0,0.0,0.265386,0.317799,...,0.0,0.0,0.0,0.0,0.768420,0.274130,0.300708,0.081951,0.0,0.412770
655,0.964603,0.0,0.475822,0.715284,0.757143,0.898200,0.0,0.0,0.265386,0.298549,...,0.0,0.0,0.0,0.0,0.737455,0.223922,0.323379,0.093785,0.0,0.543904
656,0.930971,0.0,0.475822,0.690709,0.717857,0.839722,0.0,0.0,0.265386,0.294672,...,0.0,0.0,0.0,0.0,0.673147,0.291769,0.321191,0.085645,0.0,18.180364
657,0.916975,0.0,0.489660,0.778191,0.803571,0.894935,0.0,0.0,0.278788,0.364374,...,0.0,0.0,0.0,0.0,0.818619,0.355092,0.281144,0.078994,0.0,13.045591


In [43]:
from sklearn.decomposition import PCA

df = df2_scaled.copy()
hk_cols = [c for c in df.columns if c.startswith("HK_Col")]

X_hk = df[hk_cols].to_numpy()
pca = PCA(n_components=5)  # keep 5 PCs (tune this)
hk_pca = pca.fit_transform(X_hk)

hk_pca_df = pd.DataFrame(
    hk_pca,
    columns=[f"HK_PCA_{i+1}" for i in range(hk_pca.shape[1])],
    index=df.index
)

df_no_hkcols = df.drop(columns=hk_cols)
final_reduced_df = pd.concat([df_no_hkcols, hk_pca_df], axis=1)


In [44]:
final_reduced_df

,entropy_0,entropy_1,amplitude_bottleneck_0,amplitude_wasserstein_0,betti_L1_H0,betti_L2_H0,betti_L1_H1,betti_L2_H1,landscape_L1_H0,silhouette_L1_H0,...,dim_0_std_lifetime,dim_0_carlsson_f2,dim_0_carlsson_f4,dim_1_sum_death_times,Target,HK_PCA_1,HK_PCA_2,HK_PCA_3,HK_PCA_4,HK_PCA_5
0,0.939849,0.0,0.098614,0.159809,0.167857,0.289040,0.0,0.0,0.023583,0.030416,...,0.047735,0.024870,0.000096,0.0,-36.630516,-1.805982,0.301768,-0.297524,0.076389,0.103077
1,0.452725,0.0,0.675501,0.536220,0.389286,0.389639,0.0,0.0,0.490589,0.469574,...,0.661514,0.427005,0.034430,0.0,-32.755274,-1.481494,0.408269,1.083765,0.505528,0.350648
2,0.502780,0.0,0.613014,0.488256,0.360714,0.379624,0.0,0.0,0.412899,0.384258,...,0.589904,0.377790,0.025140,0.0,-16.242944,-1.530674,0.568391,0.209674,0.068514,0.476974
3,0.861676,0.0,0.270372,0.346603,0.339286,0.458352,0.0,0.0,0.103865,0.113383,...,0.223178,0.124355,0.003521,0.0,6.891582,-0.622336,-0.163668,-0.256135,-0.051017,-0.204362
4,0.788548,0.0,0.420537,0.481523,0.453571,0.529010,0.0,0.0,0.214931,0.208197,...,0.371510,0.258185,0.028176,0.0,-2.448256,0.127135,0.344818,-0.213773,-0.169482,0.013445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
654,0.945979,0.0,0.475822,0.733742,0.767857,0.888381,0.0,0.0,0.265386,0.317799,...,0.274130,0.300708,0.081951,0.0,0.412770,1.979746,0.864549,-0.262923,-0.410258,0.120873
655,0.964603,0.0,0.475822,0.715284,0.757143,0.898200,0.0,0.0,0.265386,0.298549,...,0.223922,0.323379,0.093785,0.0,0.543904,2.176358,0.462219,-0.339052,-0.314607,-0.304715
656,0.930971,0.0,0.475822,0.690709,0.717857,0.839722,0.0,0.0,0.265386,0.294672,...,0.291769,0.321191,0.085645,0.0,18.180364,1.878095,0.539786,-0.285100,-0.313250,-0.177389
657,0.916975,0.0,0.489660,0.778191,0.803571,0.894935,0.0,0.0,0.278788,0.364374,...,0.355092,0.281144,0.078994,0.0,13.045591,1.886484,1.334275,-0.087502,-0.558219,0.654458


In [45]:
file_path = 'tda_features_from_residuals_selected_scaled_w_pca.xlsx'
final_reduced_df .to_excel(file_path, index=False)  # index=False to exclude the index column